# Spatial perturbation benchmark — run

Evaluates how well a model predicts the **spatial effect of a CRISPR perturbation**: when a gene is knocked out, how does the perturbed cell itself change (**seed**) and how does that change spread to its spatial neighbours (**propagation**)?

The same cell is never measured both perturbed and unperturbed (no paired before/after), so everything is scored at the **distribution level** with the **energy distance (E-distance)**, against a **control reference niche**. Run on the lab server (env `spatial-tumor-ai`).

## 0. What is measured (read this first)

**Two halves of a perturbation effect**
- **seed** = how the *perturbed cell itself* changes.
- **propagation** = how that change spreads to its *spatial neighbours* (the niche).

**The 2×2.** Every perturbation is scored in a grid of {seed source} × {propagation model}:

|                | baseline prop (Gaussian kernel) | learned prop (GCN) |
|----------------|---------------------------------|--------------------|
| **GT seed** (the true perturbed cell, an oracle) | `e1` | `e2` |
| **model seed** (predicted from control cells)    | `e3` | `e4` |

Each `e1..e4` is an **E-distance** between the *predicted* neighbour distribution and the *observed perturbed* neighbour distribution. Lower = better. `e4` (model seed + learned prop, aka `model+learned`) is the real, deployable score.

**Energy distance.** For two groups of cells X (predicted) and Y (observed):
`E = 2·mean‖X−Y‖ − mean‖X−X'‖ − mean‖Y−Y'‖`. It is 0 when the two clouds overlap and needs no cell-to-cell pairing — it compares whole distributions.

**Distributional readout (why predictions get per-cell noise).** A model emits one vector per cell — its conditional *mean*. Real niches are full-variance clouds (spread ~6), so scoring a near-degenerate mean-field cloud with the energy distance (a *distributional* metric) inflates it structurally, no matter how good the mean shift is. So before scoring, each predicted cell gets a sampled **control residual** added (the deterministic analogue of a generative model drawing per-cell samples): it restores realistic per-cell variance without moving the mean, so E-distance measures the predicted *shift* fairly. Residuals come only from control cells — never the observed niche — so they cannot leak. (`distributional=True`.)

**Derived numbers (computed from `e1..e4`):**
- `seed_cost = e3 − e1` — penalty for *predicting* the seed vs knowing it (same propagation). >0 means seed prediction hurts.
- `learned_value = e1 − e2` — how much the learned GCN beats the Gaussian baseline (same true seed). >0 means learned propagation helps.
- `end_to_end = e4` — the deployable score.
- `leak_ok` — leakage audit. Propagation starts from a *control reference*, so if a GT-seed cell (`e1`/`e2`) were ≈0 it would mean the model copied the observed niche (a leak). Both must be clearly > 0.

**The one comparison that matters — beat the no-effect baseline.** The 2×2 only compares the four cells to *each other*; on its own it can't tell you whether *any* of them beats doing nothing. So we add one more prediction in the **same currency** (E-distance to the observed niche):
- `e_null` = E-distance(**control niche**, observed niche) = the score of the laziest guess, '**the neighbours did not change**'. This is the bar to beat.
- `oracle` = best a *non-leaking* model could reach (perfect mean shift + control-population variance) = a ceiling.

Then the headline quantity is the **gain**:

  `gain = e_null − e_method`  — **>0** the method beats 'no effect'; **<0** it is *worse* than doing nothing; bigger = better. No ratios, no clipping. `gain(model+learned)` is the bottom line (the deployable pipeline); `gain(oracle)` shows how much signal is recoverable at all.

All of `e1..e4`, `e_null`, `oracle` are computed at a **matched sample size** (same n, same observed subsample per repeat) so the energy distance's finite-sample bias cancels and the gains are clean paired differences.

## 1. Imports

In [ ]:
import matplotlib
%matplotlib inline
from spbench.adapters import get_adapter
from spbench.config import run_benchmark
from spbench.viz import (plot_2x2, plot_baseline_gain, plot_gain_per_perturbation,
                         plot_learned_value, plot_significance_contrast)
import numpy as np, pandas as pd

## 2. Load the data
The adapter reads the raw `.h5mu` slices and normalises them into one `StandardData` (expression matrix, spatial coords, per-cell perturbation label, cell type, slice id). `max_files` limits how many slices are pooled — raise it for more cells. Swap the adapter / path here to run a different dataset.

In [ ]:
DIR='/home/yiru/database/spatial_perturbed_processed/CRISPR_based/Saunders_2025_40513557'
data = get_adapter('saunders')(DIR, max_files=4).load()
print(data.n_cells, 'cells,', data.n_genes, 'genes,',
      len(data.perturbations()), 'perturbations')

## 3. Define the evaluation set
`SIGNIFICANT` = the perturbations MC-spatial flagged as having a real spatial effect (permutation p<0.05). `NON_SIGNIFICANT` = a random sample of the **same size** from every other perturbation — a proxy negative-control group (most perturbations have no spatial signal). Evaluating both, balanced, lets us check whether the model helps *specifically* where there is signal, or just everywhere (which would mean it is only a generic smoother).

In [ ]:
SIGNIFICANT = ['Pck1','Rrbp1','Hspd1','Psmc1','Sepp1','Bcl2l1','Vcp',
               'Ass1','Pten','Rrn3','Letm1','Hspa5','Sec61b','Rngtt']
sig = [p for p in SIGNIFICANT if p in set(data.perturbations())]
others = [p for p in data.perturbations() if p not in set(SIGNIFICANT)]
rng = np.random.default_rng(0)   # fixed seed -> reproducible non-significant sample
NON_SIGNIFICANT = list(rng.choice(others, size=min(len(sig), len(others)), replace=False))
EVAL = sig + NON_SIGNIFICANT
print('evaluating', len(EVAL), '=', len(sig), 'significant +',
      len(NON_SIGNIFICANT), 'random non-significant')

## 4. Run the benchmark
For each perturbation this builds the spatial graph, defines the control reference niche and the observed propagation ground truth, then fills the 2×2 by composing three models — a **trivial seed** (control + global mean shift), a **Gaussian-kernel** baseline propagation, and a self-supervised **GCN** learned propagation — and scores every cell with the E-distance. `compare=True` also computes the no-effect baseline `e_null`, the `oracle` ceiling, and the per-method **gain = e_null − e** (see section 0). (`k` = neighbours per cell; `k_ref` = matched control cells per perturbed cell.)

In [ ]:
res = run_benchmark(data, perturbations=EVAL, k=15, k_ref=5,
                    gcn_kwargs={'hidden':64,'epochs':30})

## 5. Metric table
One row per perturbation, all matched-n E-distances to the observed niche (lower = closer to reality). `e_null` is the no-effect baseline, `oracle` the ceiling, and the four `model+...` / `GT+...` columns are the 2×2 cells. **`gain_deploy = e_null − e[model+learned]`** is the bottom line: **>0 means the deployable pipeline beats doing nothing.** Sorted by `gain_deploy` (best first). `gain_oracle` shows how much signal is recoverable at all.

In [ ]:
rows=[]
for p in EVAL:
    e=res['compare'][p]['e']; g=res['compare'][p]['gain']
    rows.append(dict(perturbation=p, sig=p in set(SIGNIFICANT),
                     e_null=e['null'], oracle=e.get('oracle'),
                     GT_base=e['GT+base'], GT_learned=e['GT+learned'],
                     model_base=e['model+base'], model_learned=e['model+learned'],
                     gain_deploy=g['model+learned'], gain_oracle=g.get('oracle'),
                     leak_ok=res['leakage_pass'][p]))
df=pd.DataFrame(rows).sort_values('gain_deploy', ascending=False); df

## 6. Result figures
The headline is whether each method beats the no-effect baseline (`gain > 0`). The figures aggregate `gain` across perturbations and, for the deployable model, break it down per gene and contrast significant vs non-significant.

### Headline — each method vs the no-effect baseline
One box (+ one point per perturbation) per method, of **`gain = e_null − e`**. The solid line at **0 is the no-effect baseline**: a method is useful only where its points sit **above** it. The dashed line is the **oracle ceiling** (best a non-leaking model could reach). One glance: does any method beat doing nothing, by how much, and how far from the ceiling.

In [ ]:
plot_baseline_gain(res).savefig('fig_gain_aggregate.png', dpi=140); None

### Per gene — deployable model vs baseline
`gain = e_null − e[model+learned]` for every perturbation, sorted, coloured by significance. **Right of the line (>0) = the deployable pipeline predicts this gene's niche better than assuming 'no effect'.** Shows *which* genes (if any) it wins on, and whether they are the MC-significant ones.

In [ ]:
plot_gain_per_perturbation(res, SIGNIFICANT).savefig('fig_gain_per_pert.png', dpi=140); None

### Diagnostic — is the learned advantage signal-specific?
`learned_value = e1 − e2` (GCN vs Gaussian, holding the oracle seed) for significant vs non-significant groups, with a sign-test. **If the significant group is not clearly higher, the GCN's edge over Gaussian is generic** (a better smoother), not specific to real spatial signal.

In [ ]:
plot_significance_contrast(res, SIGNIFICANT).savefig('fig_signal_specificity.png', dpi=140); None

### Single-gene 2×2 (explainer, not a result)
What one perturbation's 2×2 grid of E-distances looks like — useful to understand the layout. Read it together with `e_null` (the baseline) for that gene, not in isolation.

In [ ]:
plot_2x2(res['grids'][EVAL[0]], title=EVAL[0])

## 7. Bottom line
Count how many perturbations the deployable pipeline actually beats the baseline on (`gain_deploy > 0`), and whether they are the MC-significant ones. `learned_value > 0` alone does **not** prove signal-specificity — confirm with the contrast figure and a future label-permutation control.

In [ ]:
g = {p: res['compare'][p]['gain']['model+learned'] for p in EVAL}
wins = [p for p in EVAL if g[p] > 0]
print(f'{len(wins)}/{len(EVAL)} perturbations beat the no-effect baseline:', wins)
print('  of which MC-significant:', [p for p in wins if p in set(SIGNIFICANT)])